# Refresh the Lakebase synced table

Runs a SNAPSHOT refresh of the synced table `<catalog>.<schema>.pdu_latest_status_synced` (Unity Catalog -> Lakebase Postgres) by starting an update of the managed pipeline behind it, and waits for it to finish. Used by the `pdu-refresh` job right after the monitoring pipeline update.

If a sync is already running it waits for that one, then starts a fresh one so the copy reflects the latest `pdu_latest_status`.

In [0]:
%pip install -q "databricks-sdk>=0.118.0"
dbutils.library.restartPython()

In [0]:
import re

dbutils.widgets.text("synced_table", "mdk_demo_catalog.pdu_monitoring.pdu_latest_status_synced")
dbutils.widgets.text("timeout_minutes", "15")

SYNCED_TABLE = dbutils.widgets.get("synced_table").strip()
TIMEOUT_S = int(dbutils.widgets.get("timeout_minutes")) * 60
if not re.fullmatch(r"[A-Za-z0-9_]+\.[A-Za-z0-9_]+\.[A-Za-z0-9_]+", SYNCED_TABLE):
    raise ValueError(f"synced_table must be catalog.schema.table, got {SYNCED_TABLE!r}")

In [0]:
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceConflict
from databricks.sdk.service.pipelines import UpdateInfoState

w = WorkspaceClient()
pipeline_id = w.postgres.get_synced_table(name=f"synced_tables/{SYNCED_TABLE}").status.pipeline_id
print(f"{SYNCED_TABLE} -> sync pipeline {pipeline_id}")

TERMINAL = {UpdateInfoState.COMPLETED, UpdateInfoState.FAILED, UpdateInfoState.CANCELED}
deadline = time.time() + TIMEOUT_S


def wait_for(update_id: str) -> UpdateInfoState:
    while time.time() < deadline:
        state = w.pipelines.get_update(pipeline_id=pipeline_id, update_id=update_id).update.state
        if state in TERMINAL:
            return state
        time.sleep(10)
    raise TimeoutError(f"sync update {update_id} did not finish within the timeout")


while True:
    try:
        update_id = w.pipelines.start_update(pipeline_id=pipeline_id).update_id
        break
    except ResourceConflict:
        # A sync is already running: let it finish, then start a fresh one.
        active = w.pipelines.get(pipeline_id=pipeline_id).latest_updates[0]
        print(f"sync {active.update_id} already running; waiting")
        wait_for(active.update_id)

state = wait_for(update_id)
print(f"sync update {update_id}: {state.value}")
if state != UpdateInfoState.COMPLETED:
    raise RuntimeError(f"Lakebase sync update {update_id} ended in {state.value}")